# Geocode Idealista Barcelona Addresses

This notebook extracts address-like text from `../data/idealista_barcelona_sale_urls.csv`, identifies listings with a street number, and geocodes only those usable addresses. In this file, the address-like text is stored in `address_search`; if a future export contains a true `description` column, the notebook will use that first.

## Plan

1. Load the Idealista URL/listing file.
2. Use the best available address text column: `description`, then `address_search`, then `description_search`.
3. Remove the property-type prefix before `in`, e.g. `Flat / apartment in Calle de Pau Claris, 76, ...` becomes `Calle de Pau Claris, 76, ...`.
4. Flag rows that contain a street number. Rows with only a neighborhood/street name are kept but not sent for exact geocoding.
5. Geocode unique full-address candidates with OpenStreetMap Nominatim, using a local CSV cache and a one-second delay.
6. Merge lat/lon back to all properties and export both row-level results and a summary.

In [1]:
import re
import time
from pathlib import Path

import pandas as pd
import requests
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 120)

c:\Users\sffra\Downloads\BSE 2025-2026\scrape-idealista\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
INPUT_CSV = Path("../data/idealista_barcelona_sale_urls.csv")
OUTPUT_CSV = Path("../data/idealista_barcelona_sale_urls_geocoded.csv")
SUMMARY_CSV = Path("../data/idealista_barcelona_sale_urls_geocode_summary.csv")
CACHE_CSV = Path("../data/idealista_geocode_cache.csv")

# Nominatim requires a descriptive User-Agent. Replace the email/contact with yours if you run a large batch.
NOMINATIM_URL = "https://nominatim.openstreetmap.org/search"
USER_AGENT = "scrape-idealista-barcelona-geocoder/1.0 (local research notebook)"
REQUEST_DELAY_SECONDS = 1.1
MAX_ADDRESSES_TO_GEOCODE = None  # set to a small integer, e.g. 25, for a test run

df = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(df):,} property rows from {INPUT_CSV}")
df.head()

Loaded 10,619 property rows from data\idealista_barcelona_sale_urls.csv


,propertyCode,url,address_search,price_search,details_search,description_search,source_page,scraped_at,seed_url,seed_label
0,109696793,https://www.idealista.com/en/inmueble/109696793/,"Flat / apartment in Calle d'Aragó, La Dreta de l'Eixample, Barcelona","1,199,000 €",3 bed. 164 m² 1st floor exterior with lift | 3 bed. | 164 m² | 1st floor exterior with lift,FOR SALE: Amazing and bright renovated modernist apartment with terrace in the Eixample. This exceptional property F...,https://www.idealista.com/en/venta-viviendas/barcelona-barcelona/,2026-04-17T15:35:12+00:00,NaN,NaN
1,110509586,https://www.idealista.com/en/inmueble/110509586/,"Flat / apartment in Calle de Tamarit, Sant Antoni, Barcelona","545,000 €",2 bed. 80 m² 4th floor exterior with lift | 2 bed. | 80 m² | 4th floor exterior with lift,We present this completely renovated apartment of great quality the apartment consists of two bedrooms two bathrooms...,https://www.idealista.com/en/venta-viviendas/barcelona-barcelona/,2026-04-17T15:35:12+00:00,NaN,NaN
2,110630744,https://www.idealista.com/en/inmueble/110630744/,"Flat / apartment in Calle del Consell de Cent, La Dreta de l'Eixample, Barcelona","1,390,000 €",3 bed. 139 m² 3rd floor exterior with lift | 3 bed. | 139 m² | 3rd floor exterior with lift,Excellent renovated apartment with original features FOR SALE in the Eixample. This flat FOR SALE is located in a st...,https://www.idealista.com/en/venta-viviendas/barcelona-barcelona/,2026-04-17T15:35:12+00:00,NaN,NaN
3,110063350,https://www.idealista.com/en/inmueble/110063350/,"Flat / apartment in Calle de Girona, La Dreta de l'Eixample, Barcelona","2,200,000 €",3 bed. 271 m² 1st floor exterior with lift | 3 bed. | 271 m² | 1st floor exterior with lift,Stunning refurbished modernist apartment with a large terrace in Eixample. This magnificent property is located in o...,https://www.idealista.com/en/venta-viviendas/barcelona-barcelona/,2026-04-17T15:35:12+00:00,NaN,NaN
4,110182478,https://www.idealista.com/en/inmueble/110182478/,"Flat / apartment in Calle de Mallorca, La Dreta de l'Eixample, Barcelona","2,195,000 €",5 bed. 221 m² 5th floor exterior with lift | 5 bed. | 221 m² | 5th floor exterior with lift,Outstanding brand new apartment in a stately building next to Paseo de Gracia. This wonderful flat has been complete...,https://www.idealista.com/en/venta-viviendas/barcelona-barcelona/,2026-04-17T15:35:12+00:00,NaN,NaN


## Extract Address Candidates

In [3]:
SOURCE_COLUMN_PRIORITY = ["description", "address_search", "description_search"]
source_col = next((col for col in SOURCE_COLUMN_PRIORITY if col in df.columns), None)
if source_col is None:
    raise ValueError(f"None of the expected source columns exist: {SOURCE_COLUMN_PRIORITY}")

print(f"Using `{source_col}` as the source column for address extraction.")

Using `address_search` as the source column for address extraction.


In [4]:
STREET_WORD_RE = re.compile(
    r"\b("
    r"calle|carrer|avenida|avinguda|av\.?|paseo|passeig|rambla|ronda|plaza|plaça|pasaje|passatge|"
    r"travessera|carretera|via|gran via|cam[ií]|baixada|torrent|riera|jard[ií]|moll|muelle|"
    r"pla|portal|pujades|sender|camino|cami"
    r")\b",
    flags=re.IGNORECASE,
)
STREET_NUMBER_RE = re.compile(r"(?<!\d)\d{1,4}(?:\s*[-/]\s*\d{1,4})?(?:\s*[A-Za-z])?(?!\d)")
BAD_PLACEHOLDER_RE = re.compile(r"ask the advertiser|photos", flags=re.IGNORECASE)


def clean_spaces(value):
    return re.sub(r"\s+", " ", str(value)).strip()


def extract_address_candidate(value):
    if pd.isna(value):
        return ""
    text = clean_spaces(value)
    if not text or BAD_PLACEHOLDER_RE.search(text):
        return ""
    # Idealista labels usually look like: "Flat / apartment in Calle de Pau Claris, 76, ...".
    match = re.search(r"\bin\s+(.+)$", text, flags=re.IGNORECASE)
    if match:
        text = match.group(1).strip()
    return text.strip(" ,")


def has_street_word(value):
    return bool(value and STREET_WORD_RE.search(value))


def has_street_number(value):
    return bool(value and STREET_NUMBER_RE.search(value))


def build_geocode_query(address):
    if not address:
        return ""
    query = clean_spaces(address)
    if not re.search(r"\bbarcelona\b", query, flags=re.IGNORECASE):
        query = f"{query}, Barcelona"
    if not re.search(r"\b(spain|españa|espanya)\b", query, flags=re.IGNORECASE):
        query = f"{query}, Spain"
    return query


def parse_price_eur(value):
    if pd.isna(value):
        return pd.NA
    match = re.search(r"\d[\d.,]*", str(value))
    if not match:
        return pd.NA
    number_text = match.group(0).replace(".", "").replace(",", "")
    try:
        return int(number_text)
    except ValueError:
        return pd.NA


work = df.copy()
if "price_search" in work.columns:
    work["price_eur"] = work["price_search"].map(parse_price_eur).astype("Int64")
else:
    work["price_eur"] = pd.Series(pd.NA, index=work.index, dtype="Int64")
work["address_source_column"] = source_col
work["address_source_text"] = work[source_col]
work["address_candidate"] = work["address_source_text"].map(extract_address_candidate)
work["has_address_candidate"] = work["address_candidate"].ne("")
work["has_street_word"] = work["address_candidate"].map(has_street_word)
work["has_street_number"] = work["address_candidate"].map(has_street_number)
# Exact property geocoding needs a street number. `has_street_word` is kept for auditing, but not required because
# some Idealista labels omit a street-type prefix while still containing a usable numbered address.
work["geocode_eligible"] = work["has_address_candidate"] & work["has_street_number"]
work["geocode_query"] = work["address_candidate"].where(work["geocode_eligible"], "").map(build_geocode_query)

work[["propertyCode", source_col, "address_candidate", "has_street_number", "geocode_eligible", "geocode_query"]].head(20)

,propertyCode,address_search,address_candidate,has_street_number,geocode_eligible,geocode_query
0,109696793,"Flat / apartment in Calle d'Aragó, La Dreta de l'Eixample, Barcelona","Calle d'Aragó, La Dreta de l'Eixample, Barcelona",False,False,
1,110509586,"Flat / apartment in Calle de Tamarit, Sant Antoni, Barcelona","Calle de Tamarit, Sant Antoni, Barcelona",False,False,
2,110630744,"Flat / apartment in Calle del Consell de Cent, La Dreta de l'Eixample, Barcelona","Calle del Consell de Cent, La Dreta de l'Eixample, Barcelona",False,False,
3,110063350,"Flat / apartment in Calle de Girona, La Dreta de l'Eixample, Barcelona","Calle de Girona, La Dreta de l'Eixample, Barcelona",False,False,
4,110182478,"Flat / apartment in Calle de Mallorca, La Dreta de l'Eixample, Barcelona","Calle de Mallorca, La Dreta de l'Eixample, Barcelona",False,False,
5,106144229,"Flat / apartment in La Dreta de l'Eixample, Barcelona","La Dreta de l'Eixample, Barcelona",False,False,
6,109488303,"Flat / apartment in Calle de Girona, La Dreta de l'Eixample, Barcelona","Calle de Girona, La Dreta de l'Eixample, Barcelona",False,False,
7,110055339,"Flat / apartment in Calle del Comte d'Urgell, L'Antiga Esquerra de l'Eixample, Barcelona","Calle del Comte d'Urgell, L'Antiga Esquerra de l'Eixample, Barcelona",False,False,
8,111026222,"Flat / apartment in Calle de Roger de Llúria, La Dreta de l'Eixample, Barcelona","Calle de Roger de Llúria, La Dreta de l'Eixample, Barcelona",False,False,
9,110823853,"Flat / apartment in Calle de Pau Claris, 76, La Dreta de l'Eixample, Barcelona","Calle de Pau Claris, 76, La Dreta de l'Eixample, Barcelona",True,True,"Calle de Pau Claris, 76, La Dreta de l'Eixample, Barcelona, Spain"


## Coverage Summary Before Geocoding

In [5]:
summary = pd.DataFrame(
    {
        "metric": [
            "properties_pulled",
            "nonblank_source_text",
            "address_candidates_extracted",
            "street_or_place_candidates",
            "candidates_with_street_number",
            "geocode_eligible_full_addresses",
            "unique_geocode_queries",
        ],
        "count": [
            len(work),
            work["address_source_text"].notna().sum(),
            work["has_address_candidate"].sum(),
            work["has_street_word"].sum(),
            work["has_street_number"].sum(),
            work["geocode_eligible"].sum(),
            work.loc[work["geocode_eligible"], "geocode_query"].nunique(),
        ],
    }
)
summary["share_of_properties"] = summary["count"] / len(work)
summary

,metric,count,share_of_properties
0,properties_pulled,10619,1.000000
1,nonblank_source_text,10619,1.000000
2,address_candidates_extracted,10584,0.996704
3,street_or_place_candidates,7443,0.700913
4,candidates_with_street_number,1174,0.110557
5,geocode_eligible_full_addresses,1174,0.110557
6,unique_geocode_queries,884,0.083247


In [6]:
print(f"Properties pulled: {len(work):,}")
print(f"Rows with address candidates: {work['has_address_candidate'].sum():,}")
print(f"Rows with geocode-eligible street-number addresses: {work['geocode_eligible'].sum():,}")
print(f"Unique geocode queries: {work.loc[work['geocode_eligible'], 'geocode_query'].nunique():,}")

Properties pulled: 10,619
Rows with address candidates: 10,584
Rows with geocode-eligible street-number addresses: 1,174
Unique geocode queries: 884


Rows without a street number are usually not exact enough for property-level geocoding. Keep them in the output for auditing, but do not send them as exact addresses.

In [7]:
work.loc[
    work["has_address_candidate"] & ~work["geocode_eligible"],
    ["propertyCode", "address_candidate", "has_street_word", "has_street_number", "url"],
].head(25)

,propertyCode,address_candidate,has_street_word,has_street_number,url
0,109696793,"Calle d'Aragó, La Dreta de l'Eixample, Barcelona",True,False,https://www.idealista.com/en/inmueble/109696793/
1,110509586,"Calle de Tamarit, Sant Antoni, Barcelona",True,False,https://www.idealista.com/en/inmueble/110509586/
2,110630744,"Calle del Consell de Cent, La Dreta de l'Eixample, Barcelona",True,False,https://www.idealista.com/en/inmueble/110630744/
3,110063350,"Calle de Girona, La Dreta de l'Eixample, Barcelona",True,False,https://www.idealista.com/en/inmueble/110063350/
4,110182478,"Calle de Mallorca, La Dreta de l'Eixample, Barcelona",True,False,https://www.idealista.com/en/inmueble/110182478/
5,106144229,"La Dreta de l'Eixample, Barcelona",False,False,https://www.idealista.com/en/inmueble/106144229/
6,109488303,"Calle de Girona, La Dreta de l'Eixample, Barcelona",True,False,https://www.idealista.com/en/inmueble/109488303/
7,110055339,"Calle del Comte d'Urgell, L'Antiga Esquerra de l'Eixample, Barcelona",True,False,https://www.idealista.com/en/inmueble/110055339/
8,111026222,"Calle de Roger de Llúria, La Dreta de l'Eixample, Barcelona",True,False,https://www.idealista.com/en/inmueble/111026222/
10,108818765,"Calle de la Diputació, La Dreta de l'Eixample, Barcelona",True,False,https://www.idealista.com/en/inmueble/108818765/


## Geocode Unique Eligible Addresses

This uses OpenStreetMap Nominatim. For a large or repeated project, consider a paid geocoding provider or a local Nominatim instance. The local cache keeps prior results and makes reruns cheap.

In [8]:
def load_cache(path):
    if path.exists():
        cache = pd.read_csv(path)
        if "query" in cache.columns:
            cache = cache.drop_duplicates("query", keep="last")
        return cache
    return pd.DataFrame(columns=["query", "lat", "lon", "display_name", "importance", "osm_type", "osm_id", "geocode_status", "geocoded_at"])


def save_cache(cache, path):
    cache.drop_duplicates("query", keep="last").to_csv(path, index=False)


def geocode_one(query):
    params = {
        "q": query,
        "format": "jsonv2",
        "limit": 1,
        "addressdetails": 1,
        "countrycodes": "es",
    }
    response = requests.get(NOMINATIM_URL, params=params, headers={"User-Agent": USER_AGENT}, timeout=30)
    response.raise_for_status()
    results = response.json()
    if not results:
        return {
            "query": query,
            "lat": pd.NA,
            "lon": pd.NA,
            "display_name": "",
            "importance": pd.NA,
            "osm_type": "",
            "osm_id": pd.NA,
            "geocode_status": "not_found",
            "geocoded_at": pd.Timestamp.utcnow().isoformat(),
        }
    best = results[0]
    return {
        "query": query,
        "lat": float(best.get("lat")) if best.get("lat") is not None else pd.NA,
        "lon": float(best.get("lon")) if best.get("lon") is not None else pd.NA,
        "display_name": best.get("display_name", ""),
        "importance": best.get("importance", pd.NA),
        "osm_type": best.get("osm_type", ""),
        "osm_id": best.get("osm_id", pd.NA),
        "geocode_status": "found",
        "geocoded_at": pd.Timestamp.utcnow().isoformat(),
    }

In [9]:
cache = load_cache(CACHE_CSV)
cached_queries = set(cache["query"].dropna()) if len(cache) else set()

queries = sorted(work.loc[work["geocode_eligible"], "geocode_query"].dropna().unique())
queries_to_fetch = [query for query in queries if query and query not in cached_queries]
if MAX_ADDRESSES_TO_GEOCODE is not None:
    queries_to_fetch = queries_to_fetch[:MAX_ADDRESSES_TO_GEOCODE]

print(f"Unique eligible queries: {len(queries):,}")
print(f"Already cached: {len(cached_queries & set(queries)):,}")
print(f"Queries to fetch this run: {len(queries_to_fetch):,}")

Unique eligible queries: 884
Already cached: 884
Queries to fetch this run: 0


In [10]:
new_rows = []
for query in tqdm(queries_to_fetch):
    try:
        new_rows.append(geocode_one(query))
    except Exception as exc:
        new_rows.append(
            {
                "query": query,
                "lat": pd.NA,
                "lon": pd.NA,
                "display_name": "",
                "importance": pd.NA,
                "osm_type": "",
                "osm_id": pd.NA,
                "geocode_status": f"error: {type(exc).__name__}: {exc}",
                "geocoded_at": pd.Timestamp.utcnow().isoformat(),
            }
        )
    time.sleep(REQUEST_DELAY_SECONDS)

if new_rows:
    cache = pd.concat([cache, pd.DataFrame(new_rows)], ignore_index=True)
    save_cache(cache, CACHE_CSV)

cache = load_cache(CACHE_CSV)
cache.head()

0it [00:00, ?it/s]


,query,lat,lon,display_name,importance,osm_type,osm_id,geocode_status,geocoded_at
0,"Actor Morano, 2 -4, Vallvidrera - El Tibidabo i les Planes, Barcelona, Spain",41.413553,2.105543,"Carrer de l'Actor Morano, Vallvidrera, Vallvidrera, el Tibidabo i les Planes, Sarrià - Sant Gervasi, Barcelona, Barc...",0.053418,way,3.850491e+08,found,2026-04-21T09:30:31.286925+00:00
1,"Amargos, 15, El Gòtic, Barcelona, Spain",41.386430,2.174088,"15, Carrer d'Amargós, el Gòtic, Ciutat Vella, Barcelona, Barcelonès, Barcelona, Catalunya, 08002, España",0.000084,node,6.291115e+09,found,2026-04-21T09:30:33.432738+00:00
2,"Antic De Valencia, 99, El Poblenou, Barcelona, Spain",41.406771,2.203857,"Camí Antic de València, el Poblenou, Sant Martí, Barcelona, Barcelonès, Barcelona, Catalunya, 08005, España",0.071949,way,2.772931e+07,found,2026-04-21T09:30:35.671081+00:00
3,"Av Republica Argentina, 94, Vallcarca i els Penitents, Barcelona, Spain",41.410842,2.145746,"94, Avinguda de la República Argentina, Vallcarca i els Penitents, Gràcia, Barcelona, Barcelonès, Barcelona, Catalun...",0.000084,node,1.144941e+10,found,2026-04-21T09:30:37.437421+00:00
4,"Av. Sant Ramon Nonat, 11, La Maternitat i Sant Ramon, Barcelona, Spain",41.376721,2.116729,"11-13, Avinguda de Sant Ramon Nonat, Sant Ramon, la Maternitat i Sant Ramon, les Corts, Barcelona, Barcelonès, Barce...",0.000084,node,1.203992e+10,found,2026-04-21T09:30:39.232608+00:00


## Merge Geocodes and Export

In [11]:
geo_cols = ["query", "lat", "lon", "display_name", "importance", "osm_type", "osm_id", "geocode_status", "geocoded_at"]
geocoded = work.merge(cache[geo_cols], left_on="geocode_query", right_on="query", how="left").drop(columns=["query"])

geocoded["geocode_found"] = geocoded["lat"].notna() & geocoded["lon"].notna()

final_summary = pd.DataFrame(
    {
        "metric": [
            "properties_pulled",
            "rows_with_address_candidates",
            "rows_with_street_number_addresses",
            "rows_eligible_for_geocoding",
            "rows_with_lat_lon",
            "unique_eligible_queries",
            "unique_queries_with_lat_lon",
        ],
        "count": [
            len(geocoded),
            geocoded["has_address_candidate"].sum(),
            geocoded["has_street_number"].sum(),
            geocoded["geocode_eligible"].sum(),
            geocoded["geocode_found"].sum(),
            geocoded.loc[geocoded["geocode_eligible"], "geocode_query"].nunique(),
            geocoded.loc[geocoded["geocode_found"], "geocode_query"].nunique(),
        ],
    }
)
final_summary["share_of_properties"] = final_summary["count"] / len(geocoded)

geocoded.to_csv(OUTPUT_CSV, index=False)
final_summary.to_csv(SUMMARY_CSV, index=False)

print(f"Wrote row-level geocoded data to {OUTPUT_CSV}")
print(f"Wrote summary to {SUMMARY_CSV}")
final_summary

Wrote row-level geocoded data to data\idealista_barcelona_sale_urls_geocoded.csv
Wrote summary to data\idealista_barcelona_sale_urls_geocode_summary.csv


,metric,count,share_of_properties
0,properties_pulled,10619,1.000000
1,rows_with_address_candidates,10584,0.996704
2,rows_with_street_number_addresses,1174,0.110557
3,rows_eligible_for_geocoding,1174,0.110557
4,rows_with_lat_lon,214,0.020153
5,unique_eligible_queries,884,0.083247
6,unique_queries_with_lat_lon,150,0.014126


## Plot Geocoded Addresses

In [12]:
import plotly.express as px

plot_points = geocoded.loc[geocoded["geocode_found"]].copy()
plot_points["lat"] = pd.to_numeric(plot_points["lat"], errors="coerce")
plot_points["lon"] = pd.to_numeric(plot_points["lon"], errors="coerce")
plot_points["price_eur_float"] = pd.to_numeric(plot_points["price_eur"], errors="coerce")
plot_points = plot_points.dropna(subset=["lat", "lon"])

if plot_points.empty:
    print("No rows with lat/lon yet. Run the geocoding cells first, then rerun this cell.")
else:
    lon_pad = max((plot_points["lon"].max() - plot_points["lon"].min()) * 0.08, 0.01)
    lat_pad = max((plot_points["lat"].max() - plot_points["lat"].min()) * 0.08, 0.01)
    lon_range = [plot_points["lon"].min() - lon_pad, plot_points["lon"].max() + lon_pad]
    lat_range = [plot_points["lat"].min() - lat_pad, plot_points["lat"].max() + lat_pad]

    fig = px.scatter_geo(
        plot_points,
        lon="lon",
        lat="lat",
        color="price_eur_float",
        height=700,
        hover_name="address_candidate",
        hover_data={
            "propertyCode": True,
            "price_search": True,
            "price_eur_float": ":,.0f",
            "lat": ":.6f",
            "lon": ":.6f",
        },
        color_continuous_scale="Viridis",
        title="Geocoded Listings Colored by Price",
    )
    fig.update_traces(marker={"size": 7, "opacity": 0.75})
    fig.update_geos(
        projection_type="mercator",
        lonaxis_range=lon_range,
        lataxis_range=lat_range,
        showland=True,
        landcolor="#f3efe6",
        showocean=True,
        oceancolor="#d7edf7",
        showlakes=True,
        lakecolor="#d7edf7",
        showcountries=True,
        countrycolor="#aaaaaa",
        coastlinecolor="#777777",
        showframe=False,
    )
    fig.update_layout(template="plotly_white", margin={"r": 20, "t": 50, "l": 20, "b": 20})
    fig.show()

    raw_fig = px.scatter_geo(
        plot_points,
        lon="lon",
        lat="lat",
        height=700,
        hover_name="address_candidate",
        hover_data={
            "propertyCode": True,
            "price_search": True,
            "lat": ":.6f",
            "lon": ":.6f",
        },
        title="Raw Geocoded Listings: One Dot per Listing",
    )
    raw_fig.update_traces(marker={"size": 5, "opacity": 0.45, "color": "#2f6fbb"})
    raw_fig.update_geos(
        projection_type="mercator",
        lonaxis_range=lon_range,
        lataxis_range=lat_range,
        showland=True,
        landcolor="#f3efe6",
        showocean=True,
        oceancolor="#d7edf7",
        showlakes=True,
        lakecolor="#d7edf7",
        showcountries=True,
        countrycolor="#aaaaaa",
        coastlinecolor="#777777",
        showframe=False,
    )
    raw_fig.update_layout(template="plotly_white", margin={"r": 20, "t": 50, "l": 20, "b": 20})
    raw_fig.show()

In [13]:
geocoded.loc[
    geocoded["geocode_eligible"],
    ["propertyCode", "price_search", "price_eur", "address_candidate", "geocode_query", "lat", "lon", "geocode_status", "display_name", "url"],
].head(25)

,propertyCode,price_search,price_eur,address_candidate,geocode_query,lat,lon,geocode_status,display_name,url
9,110823853,"2,150,000 €",2150000,"Calle de Pau Claris, 76, La Dreta de l'Eixample, Barcelona","Calle de Pau Claris, 76, La Dreta de l'Eixample, Barcelona, Spain",NaN,NaN,not_found,NaN,https://www.idealista.com/en/inmueble/110823853/
12,110975491,"2,900,000 €",2900000,"Calle d'Anglí, 15, Les Tres Torres, Barcelona","Calle d'Anglí, 15, Les Tres Torres, Barcelona, Spain",NaN,NaN,not_found,NaN,https://www.idealista.com/en/inmueble/110975491/
13,110437931,NaN,<NA>,"Calle President Lluís Companys, 22","Calle President Lluís Companys, 22, Barcelona, Spain",NaN,NaN,not_found,NaN,https://www.idealista.com/inmueble/110437931/
14,109853125,NaN,<NA>,"Pasaje Salut, 96","Pasaje Salut, 96, Barcelona, Spain",NaN,NaN,not_found,NaN,https://www.idealista.com/inmueble/109853125/
15,110897278,NaN,<NA>,"Calle Maternitat D’elna, 3","Calle Maternitat D’elna, 3, Barcelona, Spain",NaN,NaN,not_found,NaN,https://www.idealista.com/inmueble/110897278/
16,110971923,NaN,<NA>,"Riera De La Creu 54, 56","Riera De La Creu 54, 56, Barcelona, Spain",NaN,NaN,not_found,NaN,https://www.idealista.com/inmueble/110971923/
17,110975132,NaN,<NA>,"Avenida Mas Fuster, 144","Avenida Mas Fuster, 144, Barcelona, Spain",NaN,NaN,not_found,NaN,https://www.idealista.com/inmueble/110975132/
18,110699054,NaN,<NA>,"Calle Progrés, 179","Calle Progrés, 179, Barcelona, Spain",41.225620,1.725637,found,"Carrer dels Caputxins, Centrevila, Vilanova i la Geltrú, Garraf, Barcelona, Catalunya, 08800, España",https://www.idealista.com/inmueble/110699054/
19,108917125,NaN,<NA>,"Calle Industria, 252","Calle Industria, 252, Barcelona, Spain",41.223898,1.724770,found,"Avinguda de Francesc Macià, Centrevila, Vilanova i la Geltrú, Garraf, Barcelona, Catalunya, 08800, España",https://www.idealista.com/inmueble/108917125/
20,109664462,NaN,<NA>,"Carretera Reial, 128","Carretera Reial, 128, Barcelona, Spain",41.379938,2.061631,found,"Carretera Reial, Sant Just Desvern, Baix Llobregat, Barcelona, Catalunya, 08960, España",https://www.idealista.com/inmueble/109664462/
